# 00 — Data Preparation — Machine PF30106
### Healthy-only data preparation for Anomaly Gate training
> This machine has healthy data only. No breakdown classification.
> Run this notebook FIRST before 04_Anomaly_Gate_PF30106.ipynb

In [8]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')

✅ Imports complete.


In [9]:
# ── FILE PATH ────────────────────────────────────────────────────────
# Update this path to wherever your PF30106 Excel file is located
HEALTHY_FILE = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\Datasets\MBP_ControllerData_PF30106.xlsx'

# ── SETTINGS ─────────────────────────────────────────────────────────
MACHINE_SERIAL = 'PF30106'
TIME_STEPS     = 7

ELEC_FEATS = [
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax'
]

print(f'✅ Config loaded.')
print(f'   Machine        : {MACHINE_SERIAL}')
print(f'   TIME_STEPS     : {TIME_STEPS}')

✅ Config loaded.
   Machine        : PF30106
   TIME_STEPS     : 7


In [10]:
# ── LOAD DATA ────────────────────────────────────────────────────────
df = pd.read_excel(HEALTHY_FILE)

# Filter valid vibration records
df = df[df['machineVibration'].astype(str).str.startswith('10')].copy()

# All records in this file are healthy
df['Breakdown'] = df['Breakdown'].fillna('Healthy')
df = df[df['Breakdown'] == 'Healthy'].copy()

print(f'✅ Data loaded.')
print(f'   Machine        : {MACHINE_SERIAL}')
print(f'   Total records  : {len(df)}')
print(f'   All Healthy    : True')

✅ Data loaded.
   Machine        : PF30106
   Total records  : 518
   All Healthy    : True


In [11]:
# ── FEATURE EXTRACTION ───────────────────────────────────────────────
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df], axis=1)

X_all = extract_features(df)
print(f'✅ Features extracted: {X_all.shape}  (samples x 66 features)')

✅ Features extracted: (518, 66)  (samples x 66 features)


In [12]:
# ── SCALE ────────────────────────────────────────────────────────────
# Fit scaler on all healthy data for this machine
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all.values)
print(f'✅ Scaler fitted on {len(X_scaled)} healthy records.')

✅ Scaler fitted on 518 healthy records.


In [13]:
# ── SEQUENCE CREATION ────────────────────────────────────────────────
def create_sequences(X, time_steps):
    return np.array([X[i:i+time_steps] for i in range(len(X)-time_steps)])

X_seq = create_sequences(X_scaled, TIME_STEPS)

# 80/20 split for autoencoder
split      = int(len(X_seq) * 0.8)
X_ae_train = X_seq[:split]
X_ae_test  = X_seq[split:]

print(f'✅ Sequences created.')
print(f'   Total sequences : {len(X_seq)}')
print(f'   Train           : {X_ae_train.shape}')
print(f'   Test            : {X_ae_test.shape}')

✅ Sequences created.
   Total sequences : 511
   Train           : (408, 7, 66)
   Test            : (103, 7, 66)


In [14]:
# ── SAVE ─────────────────────────────────────────────────────────────
prepared = {
    'X_ae_train'   : X_ae_train,
    'X_ae_test'    : X_ae_test,
    'num_features' : X_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'MACHINE_SERIAL': MACHINE_SERIAL,
}
with open('prepared_data_PF30106.pkl', 'wb') as f: pickle.dump(prepared, f)
with open('scaler_PF30106.pkl',        'wb') as f: pickle.dump(scaler, f)

print('✅ Saved: prepared_data_PF30106.pkl  scaler_PF30106.pkl')
print(f'   num_features : {X_seq.shape[2]}')
print(f'   TIME_STEPS   : {TIME_STEPS}')

✅ Saved: prepared_data_PF30106.pkl  scaler_PF30106.pkl
   num_features : 66
   TIME_STEPS   : 7
